In [1]:
#Install requirements
%pip install -r "../Requirements.txt"

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\xanth\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
#Import libraries
import pandas as pd 
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import scanpy as sc
from scipy.sparse import issparse
import sys
from sklearn.model_selection import train_test_split


In [3]:
#Add the src to the path
sys.path.append(os.path.abspath(os.path.join('..')))

In [4]:
#Define output paths for data and figures
output_dir = Path('../data/data_exploration')
output_dir.mkdir(exist_ok=True)

groups_dir = Path('../data/data_sets')
groups_dir.mkdir(exist_ok=True)

output_dir_figs = Path('../figures/data_exploration')
output_dir_figs.mkdir(exist_ok=True)

In [5]:
#Age map order
age_map_order = ["eclosion", "1_very_young", "2_very_young", "1_young", "2_young", "mid_young", "mid_aged", "aged"]

In [ ]:
group_files = sorted([f for f in groups_dir.glob("*.h5ad") if not f.name.startswith("NG_") and not f.name.endswith("_train.h5ad") and not f.name.endswith("_test.h5ad")])

print(f"Found {len(group_files)} group datasets")

In [ ]:
test_size = 0.2
random_state = 42

In [ ]:
#Loop to process all the datasets
from src.data_exploration_functions import stratified_split

overview_rows = []

for gfile in group_files:
    group_name = gfile.stem
    print(f"\n Group : {group_name}")

    g_dir = output_dir_figs / group_name
    g_dir.mkdir(parents=True, exist_ok=True)

    adata = sc.read_h5ad(gfile)
    print(f"Loaded {adata.n_obs:,} cells x {adata.n_vars:,} genes")

    #Visualize the distribution of the target variable (age) before split (on the full dataset)
    age_counts = adata.obs["age_map"].value_counts().reindex(age_map_order, fill_value=0)

    fig, ax = plt.subplots(figsize=(10,4))
    bars = ax.bar(range(len(age_map_order)), age_counts.values,
                  color = "#2E75B6", edgecolor="none")
    ax.set_xticks(range(len(age_map_order)))
    ax.set_xticklabels(age_map_order, fontsize=8)
    ax.set_title(f"{group_name} - target (age) distribution before split")
    ax.set_ylabel("Number od cells")
    plt.tight_layout()
    plt.savefig(g_dir/ "age_distribution_before_split.png", dpi=300)
    plt.close()
    print("Saved Age Distribution before split plot")

    #Perform stratified train/test split

    train_idx, test_idx = stratified_split(adata.obs, test_size, random_state)
    train = adata[train_idx].copy()
    test = adata[test_idx].copy()

    print(f" Split: {train.n_obs:,} train and {test.n_obs:,} test")

    train.write_h5ad(groups_dir/ f"{group_name}_train.h5ad")
    test.write_h5ad(groups_dir / f"{group_name}_test.h5ad")
    print(f"Train and test sets saved")

    #Data exploration only on training set
    #Visualize the age distribution
    fig,ax = plt.subplots(figsize=(10,4))
    train_age = train.obs["age_map"].value_counts().reindex(age_map_order, fill_value=0)
    ax.bar(range(len(age_map_order)), train_age.values, color="#2E75B6", edgecolor="none")
    ax.set_xticks(range(len(age_map_order)))
    ax.set_xticklabels(age_map_order, fontsize=8)
    ax.set_title(f"{group_name} age distribution after split")
    ax.set_ylabel("Number of cells")

    plt.tight_layout()
    plt.savefig(g_dir / "age_distribution_train_set.png", dpi=300)
    plt.close()
    print("Saved Age Distribution in the Training set plot")

    #Visualize the cell-type proportions 
    fine_counts = train.obs["annotation"].value_counts()
    fig, ax = plt.subplots(figsize=(8, max(3, 0.35*len(fine_counts))))

    ax.barh(range(len(fine_counts)), fine_counts.values, color="#028090", edgecolor="none")
    ax.set_yticks(range(len(fine_counts)))
    ax.set_yticklabels(fine_counts.index, fontsize=8)
    ax.invert_yaxis()
    ax.set_title(f"{group_name} Cell Type Composition")
    ax.set_xlabel("Number of cells")
    plt.tight_layout()
    plt.savefig(g_dir / "train_celltype_composition.png", dpi=300)
    plt.close()
    print("Saved Celltype Composition plot")

    #Visualize nUMI vs age
    fig, axes = plt.subplots(1,2, figsize=(13,4))
    train_df = pd.DataFrame({
        "age" : train.obs["Age"].values.astype(float),
        "nUMI": train.obs["nUMI"].values.astype(float)
    })
    #boxplot
    ages_sorted = sorted(train_df["age"].unique())
    bp_data = [train_df[train_df["age"]==a]["nUMI"].values for a in ages_sorted]
    bp= axes[0].boxplot(bp_data, patch_artist=True, showfliers=False, medianprops=dict(color="white", linewidth=1.5))

    for patch in bp["boxes"]:
        patch.set_facecolor("#028090"); patch.set_alpha(0.7)
    axes[0].set_xticks(range(1, len(ages_sorted)+1))
    axes[0].set_xticklabels([f"{int(a)}d" for a in ages_sorted], fontsize=8)
    axes[0].set_title(f"{group_name} Total UMI (nUMI) per age")
    axes[0].set_xlabel("Age (days)"); axes[0].set_ylabel("nUMI")
 
    # median trend line
    med = train_df.groupby("age")["nUMI"].median()
    axes[1].plot(med.index, med.values, "o-", color="#C0392B")
    axes[1].set_title(f"{group_name} Median nUMI vs age\n")
    axes[1].set_xlabel("Age (days)"); axes[1].set_ylabel("Median nUMI")
    plt.tight_layout()
    plt.savefig(g_dir/ "numi_vs_age.png", dpi=300)
    plt.close()
    print("Saved nUMI per age plot")

    #Visualize the genes sparsity
    Xtr = train.X.toarray() if issparse(train.X) else np.array(train.X)
    zero_frac = (Xtr == 0).mean(axis=1)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(zero_frac, bins=50, color="#2E75B6", edgecolor="none")
    ax.axvline(zero_frac.mean(), color="red", linestyle="--",label=f"Mean {zero_frac.mean():.1%}")
    ax.set_title(f"{group_name} Sparsity per cell")
    ax.set_xlabel("Fraction of genes = 0")
    ax.set_ylabel("Number of cells")
    ax.legend()
    plt.tight_layout()
    plt.savefig(g_dir/ "train_sparsity.png", dpi=300)
    plt.close()
    print("Saved Sparcity in Train set plot ")
    
    #collect overview row
    overview_rows.append({
        "group": group_name,
        "total_cells": adata.n_obs,
        "train_cells": train.n_obs,
        "test_cells": test.n_obs,
        "weakest_train_bin": train_age.idxmin(),
        "weakest_train_count": int(train_age.min()),
        "n_fine_clusters": adata.obs["annotation"].nunique(),
        })
overview = pd.DataFrame(overview_rows)
overview.to_csv(output_dir / "split_overview.csv", index=False)
print("Split overview for all groups is sompleted")

In [7]:
#Select High Variable Gnes (HVGs) on the training set and apply them on the test set for each family group

n_HVGs = 2000

group_names = ["Clock", "Glia", "Kenyon_cells", "Monoaminergic", "Optic_lobe", "Peptidergic"]

for group in group_names:
    train = sc.read_h5ad(groups_dir / f"{group}_train.h5ad")
    test = sc.read_h5ad(groups_dir / f"{group}_test.h5ad")

    #Select HVGs on training set
    sc.pp.highly_variable_genes(train, n_top_genes=n_HVGs, flavor="seurat_v3")
    hvg_genes = train.var_names[train.var["highly_variable"]].tolist()

    #Apply the same HVGs list to both train and test set
    train_hvg =train[:, hvg_genes].copy()
    test_hvg = test[:, hvg_genes].copy()

    #Save the HVG-filtered datasets and the gene list
    train_hvg.write_h5ad(groups_dir / "HVGs" / f"{group}_train_hvg.h5ad")
    test_hvg.write_h5ad(groups_dir / "HVGs" / f"{group}_test_hng.h5ad")

    pd.Series(hvg_genes).to_csv(groups_dir / "HVGs" / f"{group}_hng_list.csv", index=False, header=["gene"])

    print("Saved HVG-filtered train/test sets and gene list")

c:\Users\xanth\AppData\Local\Programs\Python\Python310\lib\site-packages\legacy_api_wrap\__init__.py:88: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


Saved HVG-filtered train/test sets and gene list


c:\Users\xanth\AppData\Local\Programs\Python\Python310\lib\site-packages\legacy_api_wrap\__init__.py:88: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


Saved HVG-filtered train/test sets and gene list


c:\Users\xanth\AppData\Local\Programs\Python\Python310\lib\site-packages\legacy_api_wrap\__init__.py:88: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


...

Saved HVG-filtered train/test sets and gene list


c:\Users\xanth\AppData\Local\Programs\Python\Python310\lib\site-packages\legacy_api_wrap\__init__.py:88: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


Saved HVG-filtered train/test sets and gene list


c:\Users\xanth\AppData\Local\Programs\Python\Python310\lib\site-packages\legacy_api_wrap\__init__.py:88: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


Saved HVG-filtered train/test sets and gene list
